In [2]:
import sqlite3

con = sqlite3.connect('../volumes/data/app.db')
cur = con.cursor()

def dict_factory(cursor, row):
    d = {}
    for idx, col in enumerate(cursor.description):
        d[col[0]] = row[idx]
    return d

con.row_factory = dict_factory

In [ ]:
cur.execute("SELECT * FROM users WHERE username = ?", ('testuser',)).fetchone()

In [30]:
cur.execute("CREATE TABLE IF NOT EXISTS users (userID TEXT PRIMARY KEY, username TEXT UNIQUE, hashedPassword TEXT)").fetchall()
con.commit()

In [29]:
cur.execute("DROP TABLE IF EXISTS users")
con.commit()

In [25]:
cur.execute('PRAGMA table_info(users)').fetchall()

[(0, 'id', 'TEXT', 0, None, 1),
 (1, 'username', 'TEXT', 0, None, 0),
 (2, 'hashed_password', 'TEXT', 0, None, 0)]

In [24]:
cur.execute('PRAGMA table_info(tokens)').fetchall()

[(0, 'id', 'INTEGER', 0, None, 1),
 (1, 'userID', 'INTEGER', 0, None, 0),
 (2, 'token', 'TEXT', 0, None, 0),
 (3, 'tokenType', 'TEXT', 0, None, 0),
 (4, 'createdAt', 'TIMESTAMP', 0, 'CURRENT_TIMESTAMP', 0)]

In [5]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

[('test',), ('users',), ('sessions',), ('responses',), ('citations',)]

In [15]:
tables = cur.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
for table in tables:
    print(table[0])

users
sessions
responses
citations


In [31]:
def create_user(userID: str, username: str, hashed_password: str):
    try:
        cur.execute("INSERT INTO users (userID, username, hashedPassword) VALUES (?, ?, ?)", (userID, username, hashed_password))
        con.commit()
        return {"status": "success", "message": f"User '{username}' with ID '{userID}' added successfully."}
    except sqlite3.IntegrityError:
        return {"status": "warning", "message": f"User '{username}' with ID '{userID}' already exists."}